# 14 · Export for the Web App

**Purpose.** Package the ensemble, thresholds and metrics into the bundle the FastAPI app loads, and test it end to end.

| | |
|---|---|
| **Reads** | `best_models.json`, `test_results.json`, `open_set.json` |
| **Writes** | `models/deploy/{member_*.pt, model_card.json}` |
| **Run time (Colab T4)** | ~3 min |
| **Next notebook** | the web app (docs/WEB_APP_GUIDE.md) |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

import torch
from dermalens.export import load_bundle, measure_latency, save_bundle
from dermalens.train import model_from_checkpoint

members = load_json(paths.results / "best_models.json")["members"]
test_results = load_json(paths.results / "test_results.json")
open_set = load_json(paths.results / "open_set.json")

In [ ]:
bundle_members = []
for member, temperature in zip(members, test_results["temperatures"]):
    model, facts = model_from_checkpoint(paths.root / member["checkpoint"], "cpu")
    bundle_members.append({"model": model, "name": member["model"], "image_size": facts["image_size"],
                           "temperature": temperature})
metrics = {k: v["value"] for k, v in test_results["headline_known_classes"].items()}
metrics.update({"open_set_auroc": round(open_set["open_set_auroc"], 4),
                "grouped_accuracy": test_results["grouped_taxonomy"]["accuracy"],
                "test_images": test_results["test_images_known"]})
bundle_dir = save_bundle(bundle_members, paths.models / "deploy", CLASSES, metrics,
                         confidence_threshold=open_set["confidence_threshold"],
                         min_skin_ratio=open_set["min_skin_ratio"], color_constancy=COLOR,
                         notes=f"Split seed {info['split_seed']}; Healthy = MCSI, Psoriasis = DermaCon-IN only.")
for file in sorted(bundle_dir.iterdir()):
    print(f"{file.name:<18} {file.stat().st_size / 1e6:.1f} MB")
print(metrics)

## Reload as the app does, check speed and one full prediction

In [ ]:
reloaded, card = load_bundle(bundle_dir, "cpu")
torch.set_num_threads(2)
for model, member in reloaded:
    print(member["model_name"], measure_latency(model, member["image_size"], runs=10))

sys.path.insert(0, str(CODE_DIR / "app"))
from inference import Predictor
from dermalens.data import load_rgb
predictor = Predictor(bundle_dir)
sample = val.iloc[0]
regions = [r for r in str(sample.get("meta_regions") or "").split("|") if r and r != "nan"]
answer = predictor.predict(load_rgb(IMAGE_ROOT / sample[PATH_COL]),
                           age_group=sample.get("meta_age") if isinstance(sample.get("meta_age"), str) else None,
                           sex=sample.get("meta_sex") if isinstance(sample.get("meta_sex"), str) else None,
                           body_regions=regions)
print("True label:", sample["label"])
print({k: v for k, v in answer.items() if k not in ("heatmap_png_base64", "condition")})

**Next:** copy `models/deploy/` into the repository's `models/deploy/` and start the site:
`uvicorn app.api:app --port 8000` (see `docs/WEB_APP_GUIDE.md`).